# Library

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.feature_selection import RFE as rfe, SequentialFeatureSelector as sfs, chi2, SelectKBest as skb, VarianceThreshold as vt

from sklearn.preprocessing import StandardScaler as ss, OneHotEncoder as ohe, MinMaxScaler as mms
from sklearn.linear_model import LogisticRegression as lr, Lasso as l1
from sklearn.ensemble import RandomForestClassifier as rfc
from sklearn.tree import DecisionTreeClassifier as dtc
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split as tts, GridSearchCV as gscv, RandomizedSearchCV as rscv
from sklearn.compose import ColumnTransformer as ct
from sklearn.metrics import accuracy_score as AS, classification_report as cr, confusion_matrix as cm, recall_score as rs, precision_score as ps
from sklearn.cluster import KMeans as kmc, DBSCAN as dbsc, MeanShift as msc, AgglomerativeClustering as ac, estimate_bandwidth as eb
from sklearn.decomposition import PCA as pca, KernelPCA as kpca
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as lda, QuadraticDiscriminantAnalysis as qda
import scipy.cluster.hierarchy as sch

from skopt import BayesSearchCV as bscv
from skopt.space import Real as real, Categorical as cat, Integer as integer
from scipy.stats import randint, uniform


# Data

In [2]:
data = pd.read_csv(r"E:\DATA FOR TEST\All Codes\HPO\breast_cancer.csv")
df = data.copy()
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

In [3]:
# Drop ID
df.drop('id', axis=1, inplace=True)
df.dropna(axis=1, inplace=True)

In [4]:
value_count = round(df.diagnosis.value_counts()/len(df.diagnosis)*100, 2)
print(value_count)

# Split x and y
x = df.drop('diagnosis', axis=1)
y = data['diagnosis'].map({'B':0, 'M':1})

diagnosis
B    62.74
M    37.26
Name: count, dtype: float64


In [5]:
# Split in Training and Testing Data
x_train, x_test, y_train, y_test = tts(x, y, test_size=0.2, random_state=42, stratify=y)

# Feature Scaling
Scaler = ss()
x_train = Scaler.fit_transform(x_train)
x_test = Scaler.transform(x_test)

# Model

In [6]:
# Random Forest Classifier Model
rfc_model = rfc(
    n_estimators=500,
    random_state=42,
    max_samples=0.5,
    n_jobs=-1
)
rfc_model.fit(x_train, y_train)

# Model Prediction
y_pred = rfc_model.predict(x_test)

# Model Results
print("Accuracy Score :\t", round(AS(y_test, y_pred)*100, 2))
print("\nClassification Reports :\n", cr(y_test, y_pred))
print("Confusion Matrix :\n", cm(y_test, y_pred))

Accuracy Score :	 96.49

Classification Reports :
               precision    recall  f1-score   support

           0       0.95      1.00      0.97        72
           1       1.00      0.90      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.95      0.96       114
weighted avg       0.97      0.96      0.96       114

Confusion Matrix :
 [[72  0]
 [ 4 38]]


# Mannual HPO

In [7]:
# Estimator List
n_estimators_list = [3, 10, 50, 100, 150, 200, 500, 1000]

for EstLis in n_estimators_list:
    model = rfc(n_estimators = EstLis, random_state=42)
    model.fit(x_train, y_train)

    y_pred = model.predict(x_test)
    results = AS(y_test, y_pred)
    print(f'\nEstimator Values with Accuracy Score {EstLis} : {results}')


Estimator Values with Accuracy Score 3 : 0.9736842105263158

Estimator Values with Accuracy Score 10 : 0.9649122807017544

Estimator Values with Accuracy Score 50 : 0.9736842105263158

Estimator Values with Accuracy Score 100 : 0.9736842105263158

Estimator Values with Accuracy Score 150 : 0.9649122807017544

Estimator Values with Accuracy Score 200 : 0.9649122807017544

Estimator Values with Accuracy Score 500 : 0.9736842105263158

Estimator Values with Accuracy Score 1000 : 0.9736842105263158


In [8]:
# min Samples 
leaf_size = [1, 2, 3, 4, 5, 6, 7, 10]

for i in leaf_size:
    model = rfc(max_samples = i, random_state=42)
    model.fit(x_train, y_train)

    y_pred = model.predict(x_test)
    results = AS(y_test, y_pred)
    print(f'\nMinimum Samples with Accuracy Score {i} : {results}')


Minimum Samples with Accuracy Score 1 : 0.631578947368421

Minimum Samples with Accuracy Score 2 : 0.6754385964912281

Minimum Samples with Accuracy Score 3 : 0.8508771929824561

Minimum Samples with Accuracy Score 4 : 0.868421052631579

Minimum Samples with Accuracy Score 5 : 0.8859649122807017

Minimum Samples with Accuracy Score 6 : 0.8947368421052632

Minimum Samples with Accuracy Score 7 : 0.8947368421052632

Minimum Samples with Accuracy Score 10 : 0.8947368421052632


In [9]:
# min Samples 
leaf_size = [1, 2, 3, 4, 5, 6, 7, 10]

for i in leaf_size:
    model = rfc(max_samples = i, random_state=42, n_estimators=3)
    model.fit(x_train, y_train)

    y_pred = model.predict(x_test)
    results = AS(y_test, y_pred)
    print(f'\nMinimum Samples with Accuracy Score {i} : {results}')


Minimum Samples with Accuracy Score 1 : 0.631578947368421

Minimum Samples with Accuracy Score 2 : 0.631578947368421

Minimum Samples with Accuracy Score 3 : 0.7807017543859649

Minimum Samples with Accuracy Score 4 : 0.7017543859649122

Minimum Samples with Accuracy Score 5 : 0.7017543859649122

Minimum Samples with Accuracy Score 6 : 0.8596491228070176

Minimum Samples with Accuracy Score 7 : 0.9122807017543859

Minimum Samples with Accuracy Score 10 : 0.9122807017543859


# RandomizedSearchCV HPO

In [8]:
# Full Night Code

n_estimators = [int(x) for x in np.linspace(50, 800, 21)]
criterion = ["gini", "entropy"]
max_depth = [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20, 25, 30]
min_samples_split = list(range(2, 12))
min_samples_leaf = [int(x) for x in np.linspace(1, 9, 5)]
min_weight_fraction_leaf = [0.0, 0.01, 0.02, 0.05, 0.1]
max_features = ["sqrt", "log2"]
max_samples = [x for x in np.linspace(0.58125, 0.5833333333333334, 25)]
max_leaf_nodes = [None, 5, 10, 20, 50, 100]
min_impurity_decrease = [0.0, 1e-4, 1e-3, 1e-2, 0.1]
oob_score = [True, False]
n_jobs = [-1, 1, 2, 4]
verbose = [0, 1, 2]
warm_start = [True, False]
class_weight = [None, "balanced", "balanced_subsample"]
ccp_alpha = [0.0, 1e-4, 1e-3, 1e-2, 0.1]

# Create a Grid by This lists

grid = {
    "n_estimators": randint(50, 801),
    "criterion": ["gini", "entropy"],
    "max_depth": randint(1, 31),
    "min_samples_split": randint(2, 12),
    "min_samples_leaf": randint(1, 10),
    "min_weight_fraction_leaf": uniform(0.0, 0.1),
    "max_features": ["sqrt", "log2"],
    "max_samples": uniform(0.58125, 0.00208),
    "max_leaf_nodes": [None, 5, 10, 20, 50, 100],
    "min_impurity_decrease": uniform(0.0, 0.1),
    "oob_score": [True, False],
    "n_jobs": [-1],
    "verbose": [0],
    "warm_start": [False],
    "class_weight": [None, "balanced", "balanced_subsample"],
    "ccp_alpha": uniform(0.0, 0.1)
}


# Use Random Forest For Best Model
rscv_grid = rscv(
    estimator= rfc(random_state=42, bootstrap=True),
    cv = 3,
    n_jobs = -1,
    param_distributions = grid,
    n_iter = 1000
)
rscv_grid.fit(x_train, y_train)

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'ccp_alpha': <scipy.stats....002895AECCA70>, 'class_weight': [None, 'balanced', ...], 'criterion': ['gini', 'entropy'], 'max_depth': <scipy.stats....002895AEB0BF0>, ...}"
,n_iter,1000
,scoring,None
,n_jobs,-1
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


In [10]:
n_estimators = [int(x) for x in np.linspace(100, 1000, 10)]
max_depth = [int(x) for x in np.linspace(10, 110, 11)]
min_sample_split = [2, 3, 4, 5, 8, 10, 20, 50, 100, 200]
min_samples_leaf = [1, 2, 4, 10, 20, 50, 100]
bootstrap = [True, False]

# Create Random Grid
random_grid = {
    'n_estimators':n_estimators,
    'max_depth':max_depth,
    'min_samples_split':min_sample_split,
    'bootstrap':bootstrap,
    'min_samples_leaf':min_samples_leaf
}

# Use Random Grid for Best Hyper Parameter
rf_random = rscv(
    estimator = rfc(random_state=42),
    param_distributions = random_grid,
    n_iter = 100,
    cv = 3,
    n_jobs = -1
)
rf_random.fit(x_train, y_train)

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'bootstrap': [True, False], 'max_depth': [10, 20, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 3, ...], ...}"
,n_iter,100
,scoring,None
,n_jobs,-1
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


In [11]:
# Creat Function For Results
def evaluate(model, test_features, test_labels):
    predictions = model.predict(test_features)
    accuracy = AS(test_labels, predictions)
    print(f'Model Performance')
    print('Accuracy', round(accuracy*100, 3))
    return accuracy

In [12]:
# Basic Model
base_model = rfc(n_estimators=5, random_state=42)
base_model.fit(x_train, y_train)

base_model_accuracy = evaluate(base_model, x_test, y_test)

Model Performance
Accuracy 96.491


In [13]:
# RandomizedSearchCV Model
best_random = rf_random.best_estimator_
print(best_random)

random_accuracy = evaluate(best_random, x_test, y_test)

RandomForestClassifier(bootstrap=False, max_depth=40, n_estimators=800,
                       random_state=42)
Model Performance
Accuracy 96.491


In [14]:
print(f'Improvement : {(random_accuracy-base_model_accuracy)*100}')

Improvement : 0.0


# GridSearchCV

In [68]:
# Create Grid Serach on The Results of Random Search
param_grid = {
    'bootstrap' : [True, False],
    'max_depth' : [75, 80, 85, 90, 95, 100],
    'max_features' : [2, 3, 4, 5],
    'min_samples_split' : [8, 10, 12],
    'min_samples_leaf' : [1, 2, 3],
    'n_estimators' : [200, 250, 300, 350, 360, 400, 450, 500]
}

# Create a Base Model with Initiate GridSearchCV
grid_search = gscv(
    estimator = rfc(random_state = 42),
    param_grid = param_grid,
    cv = 3,
    n_jobs = -1
)

In [ ]:
# Fit The Data To GridSearchCV
grid_search.fit(x_train, y_train)
best_parameters = grid_serach.best_params_
print(best_parameters)

In [ ]:
# Estimate Results of Model
best_grid = grid_search.best_estimator_
print(best_grid)

grid_accuracy = evaluate(best_grid, x_test, y_test)

In [ ]:
print(f'Improvement : {(grid_accuracy-base_model_accuracy)*100}')

# Baysian HyperParameterOptimization

In [22]:
# 1. Define the Search Space (Use Ranges, not Lists)
search_spaces = {
    'n_estimators': integer(100, 1000),
    'max_depth': integer(10, 110),
    'min_samples_split': integer(2, 200),  # Note: Correct spelling!
    'min_samples_leaf': integer(1, 100),
    'bootstrap': cat([True, False])
}

# 2. Setup Bayesian Optimization
opt = bscv(
    estimator = rfc(random_state=42),
    search_spaces=search_spaces,
    n_iter=32,   # How many times to train (32 is usually good enough)
    cv=3,        # Cross-validation folds
    n_jobs=-1,   # Use all CPU cores
    random_state=42
)

In [23]:
# 3. Fit the model
# (It might take a moment to start as it calculates the first few points)
opt.fit(x_train, y_train)

# 4. Check results
print("Best Parameters:", opt.best_params_)
print("Best Score:", opt.best_score_)

Best Parameters: OrderedDict({'bootstrap': False, 'max_depth': 36, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 403})
Best Score: 0.960410131288486
